**Content Produced by UF Signal Processing Society**

**Authors: Raul Valle & Contributors**

# Beyond Kalman: EKF, UKF & Particle Filters

The [Kalman filter](./Intro_AdFilt_KF.ipynb) is optimal for linear dynamics and Gaussian noise. Reality is neither. Three sessions on the escalation ladder: linearize (EKF), sample deterministically (UKF), sample massively (particle filter) — each demonstrated on problems where its predecessor fails.

## 1. Pre-requisites

[Adaptive Filtering: Kalman](./Intro_AdFilt_KF.ipynb) — this workshop assumes its notation and predict/update instincts.

In [1]:
import numpy as np
import matplotlib.pyplot as plt
rng = np.random.default_rng(2)

---
### 🕐 Session 1 of 3 — *The Extended Kalman Filter* (~35 min)
**Goal:** linearize the model at the current estimate; track a pendulum.
**Builds on:** [Kalman workshop](./Intro_AdFilt_KF.ipynb). &nbsp; **Feeds into:** Session 2 (UKF).

---

<details>
<summary>🎓 <b>Teacher notes — Session 1: The Extended Kalman Filter</b></summary>

**Timing (~35 min).** 5 min what breaks when $f$ is nonlinear · 10 min the Taylor patch and where each approximation goes · 12 min the pendulum demo · 8 min when EKF lies.

**Board first.** Draw a curve, mark a point, draw the tangent. Then draw a narrow Gaussian at that point and a wide one. On the narrow one the tangent and the curve are indistinguishable; on the wide one they diverge badly. That single picture predicts everything about when EKF works and when it fails, and it sets up Session 2's banana demo before the room has met it.

**The asymmetry students miss.** EKF does *not* linearize everything. The mean is propagated through the true nonlinear $f$; only the covariance is propagated through the Jacobian. Point at the two lines in the code — `x_e = f(x_e)` and `P = Fj @ P @ Fj.T + Q` — and make someone say which uses which. The distinction matters because it explains why EKF's mean estimate is often decent while its *uncertainty* is quietly wrong, which is the more dangerous failure.

**Misconception.** "EKF is the Kalman filter for nonlinear systems." It is not, in the sense that matters: the Kalman filter is provably optimal under its assumptions, while EKF is a heuristic with no optimality guarantee at all. It can and does diverge — a bad linearization produces a bad covariance, which produces a bad gain, which produces a worse estimate and a worse linearization next step. Say plainly that EKF divergence is a real engineering phenomenon, not a hypothetical.

**Ask the room.** "We only measure $\sin\theta$. Why can this work at all, given $\sin$ is not invertible?" The answer is the whole point of state estimation: a single measurement is ambiguous, but the *dynamics* disambiguate — the filter knows the pendulum's momentum and where it was going. Then point at the `arcsin(z)` scatter on the plot, which is what you get without dynamics, and let the contrast land.

**Note the deliberately hard setup.** The initial angle is 2.2 rad, far outside the small-angle regime where $\sin\theta \approx \theta$, so this is genuinely nonlinear rather than a linear problem in disguise. The initial guess `[1.5, 0.5]` is also deliberately wrong. Both choices are there so the filter has to work; mention it, or students will assume the demo was rigged easy.

**If the demo misbehaves.** Push the initial guess much further from truth (try `[0.0, 0.0]`) and the EKF can lock onto the wrong phase of the swing and never recover — an excellent live demonstration of divergence if you have five spare minutes, and a better lesson than a clean run.
</details>

## 2. EKF: Pretend It's Linear (Locally)

💡 **Intuition.** Nonlinear dynamics $f(\mathbf{x})$ break the Kalman derivation. EKF's fix is a first-order Taylor patch: propagate the *mean* through the true $f$, but propagate the *covariance* through $f$'s Jacobian at the current estimate — a fresh linearization every step. It works beautifully while the uncertainty stays small enough that $f$ is locally straight; it lies when curvature bites within one standard deviation.

In [2]:
# Pendulum: x = [angle, angular velocity], observe only sin(angle) (e.g. a horizontal position sensor)
dt, g_l = 0.02, 9.81
def f(x):  return np.array([x[0] + dt*x[1], x[1] - dt*g_l*np.sin(x[0])])
def F_jac(x): return np.array([[1, dt], [-dt*g_l*np.cos(x[0]), 1]])
def h(x):  return np.array([np.sin(x[0])])
def H_jac(x): return np.array([[np.cos(x[0]), 0.0]])

T = 600
Q = np.diag([1e-6, 1e-4]); R_m = np.array([[0.02**2]])
xs = np.zeros((T, 2)); xs[0] = [2.2, 0]                     # LARGE swing: genuinely nonlinear
for t in range(1, T):
    xs[t] = f(xs[t-1]) + rng.multivariate_normal([0,0], Q)
zs = np.array([h(x) + rng.normal(0, 0.02, 1) for x in xs])

x_e, P = np.array([1.5, 0.5]), np.eye(2)                    # wrong initial guess
est = []
for t in range(T):
    Fj = F_jac(x_e)
    x_e = f(x_e); P = Fj @ P @ Fj.T + Q                     # predict through TRUE f, JACOBIAN for P
    Hj = H_jac(x_e)
    S = Hj @ P @ Hj.T + R_m
    K = P @ Hj.T @ np.linalg.inv(S)
    x_e = x_e + (K @ (zs[t] - h(x_e))).ravel()
    P = (np.eye(2) - K @ Hj) @ P
    est.append(x_e.copy())
est = np.array(est)

plt.figure(figsize=(9, 2.8))
plt.plot(xs[:, 0], "k--", linewidth=1, label="true angle")
plt.plot(est[:, 0], label="EKF estimate")
plt.plot(np.arcsin(np.clip(zs, -1, 1)), ".", markersize=2, alpha=0.3, label="naive arcsin(z)")
plt.legend(); plt.title("EKF tracks a large-swing pendulum from sin(θ) alone")
plt.tight_layout(); plt.show()
print(f"EKF angle RMSE: {np.sqrt(np.mean((est[100:,0]-xs[100:,0])**2)):.4f} rad")

EKF angle RMSE: 0.0082 rad


/tmp/ipykernel_2045392/3642640616.py:33: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.tight_layout(); plt.show()


**What just happened.** A steady-state angle RMSE of **0.0082 rad** — about half a degree — from a filter that started at `[1.5, 0.5]` when the truth was `[2.2, 0]`, and that never observes the angle directly. Compare against the faint `arcsin(z)` scatter: that is the same measurement stream decoded without any dynamics, and it is both far noisier and outright wrong wherever $|\theta| > \pi/2$, because $\arcsin$ cannot tell $\theta$ from $\pi - \theta$.

That contrast is the argument for state estimation in one plot. A single measurement of $\sin\theta$ is genuinely ambiguous, and no amount of cleverness resolves it instant by instant. The filter succeeds because it fuses the measurement with a *model* — it knows the pendulum has momentum and roughly where it was heading, so the impossible branch is inconsistent with everything seen so far. Information from the past is doing at least as much work here as the current sample.

**Where the approximation actually sits.** The mean went through the true nonlinear $f$; only the covariance went through the Jacobian, and only that step is approximate. This is why EKF's point estimate can look excellent while its reported uncertainty is subtly wrong — and the reported uncertainty is what drives the gain. The failure mode is therefore quiet: nothing throws, the plot looks fine, and the filter is overconfident.

**And this problem is friendlier than it appears.** The angle uncertainty stays small once the filter has locked on, so within any one step $\sin$ is nearly straight across the width of the belief, and the tangent-line patch is a good one. Widen the belief — a worse initial guess, a larger $Q$, or sparser measurements — and curvature starts to bite inside a single standard deviation, at which point the Jacobian stops describing what the distribution actually does. Session 2 builds exactly that case and shows the tangent line failing in a way no amount of tuning fixes.

---
### 🕐 Session 2 of 3 — *The Unscented Kalman Filter* (~35 min)
**Goal:** replace Jacobians with sigma points; win when curvature bites.
**Builds on:** Session 1. &nbsp; **Feeds into:** Session 3 (particle filters).

---

<details>
<summary>🎓 <b>Teacher notes — Session 2: The Unscented Kalman Filter</b></summary>

**Timing (~35 min).** 8 min the motto and the sigma-point construction · 12 min the banana demo · 8 min why second-order accuracy for the same cost · 7 min when UKF is *not* the answer.

**Lead with the motto, it does real work.** *It is easier to approximate a distribution than a nonlinear function.* EKF tries to approximate $f$ (by its tangent) and then pushes the distribution through the approximation. UKF leaves $f$ alone — every sigma point goes through the **true** $f$ — and approximates the distribution instead, by a handful of well-chosen points. Students who hold that sentence can reconstruct the whole method.

**Board first.** Draw the curve and a wide Gaussian from Session 1 again. Mark five points at the mean and at $\pm$ the covariance directions, map each along the curve by hand, and show the mapped points bending away from the tangent line. Then say: refit a Gaussian to *those*. That is the unscented transform, and it takes ninety seconds to draw.

**Sigma points are not random.** This is the misconception to pre-empt. They are deterministically placed, there are only $2n+1$ of them, and running the cell twice gives identical answers — nothing is being sampled. Students hear "points" and think Monte Carlo, which then makes Session 3's particle filter look like the same idea with more points. It is not: sigma points are a quadrature rule, particles are a sampled representation of an arbitrary distribution.

**The cost argument that sells UKF.** For an $n$-dimensional state, UKF needs $2n+1$ evaluations of $f$; EKF needs one evaluation plus an $n \times n$ Jacobian. Those are comparable, and the Jacobian is often the expensive, error-prone, human part — it may not exist in closed form at all if $f$ is a lookup table, a black-box simulator, or a chunk of legacy code. So UKF is usually *second-order accurate for first-order cost, with no derivatives to derive*. That is the pitch, and it is why UKF is the sensible default in practice.

**Ask the room.** Before running: "the range noise is tiny (σ = 0.02) but the angle noise is large (σ = 0.35 rad, about 20°). What shape is the cloud after converting to Cartesian?" Get them to predict a banana — an arc, not an ellipse. Then the punchline: no Gaussian, however well fitted, has a banana's mean *on* the banana. The EKF's answer is not merely imprecise, it is systematically pushed outward.

**Point at the closed form.** The debrief works out that the exact answer is $e^{-\sigma^2/2} = 0.9406$. Worth showing, because it upgrades the demo from "UKF agrees with Monte Carlo" to "UKF agrees with the exact answer, and we can say why EKF is biased by 5.9%." If the room enjoys it, $E[\sin\theta]$ for Gaussian $\theta$ is a one-line exercise.

**Be honest about the limits.** UKF still summarises the belief as one mean and one covariance. It handles curvature; it cannot represent two hypotheses at once. That limitation is precisely Session 3's opening.
</details>

## 3. UKF: Sample the Belief, Not the Slope

💡 **Intuition.** EKF pushes *one* point and a slope through $f$. The unscented transform pushes a handful of **sigma points** — deterministically placed at the mean ± scaled covariance directions — through the *true* nonlinear $f$, then refits mean and covariance to where they landed. No Jacobians (great when $f$ is ugly or black-box), and accurate to second order instead of first. Motto: *it's easier to approximate a distribution than a nonlinear function.*

In [3]:
# The classic curvature demo: push a Gaussian through polar→Cartesian
def polar_to_xy(p): return np.array([p[0]*np.cos(p[1]), p[0]*np.sin(p[1])])
mean_p = np.array([1.0, np.pi/2]); cov_p = np.diag([0.02**2, 0.35**2])   # tight range, WIDE angle

# Monte Carlo truth
samples = rng.multivariate_normal(mean_p, cov_p, 4000)
xy = np.array([polar_to_xy(s) for s in samples])
mc_mean = xy.mean(0)

# EKF-style: linearize at the mean
J = np.array([[np.cos(mean_p[1]), -mean_p[0]*np.sin(mean_p[1])],
              [np.sin(mean_p[1]),  mean_p[0]*np.cos(mean_p[1])]])
ekf_mean = polar_to_xy(mean_p)

# Unscented transform
n_dim, kappa = 2, 1.0
L = np.linalg.cholesky((n_dim + kappa) * cov_p)
sigmas = [mean_p] + [mean_p + L[:, i] for i in range(2)] + [mean_p - L[:, i] for i in range(2)]
Wts = np.array([kappa/(n_dim+kappa)] + [1/(2*(n_dim+kappa))]*4)
mapped = np.array([polar_to_xy(s) for s in sigmas])
ut_mean = Wts @ mapped

plt.figure(figsize=(5, 4))
plt.scatter(xy[:, 0], xy[:, 1], s=2, alpha=0.15, label="truth (Monte Carlo)")
plt.plot(*mc_mean, "k*", markersize=14, label=f"true mean")
plt.plot(*ekf_mean, "rs", markersize=9, label="EKF (linearized): biased outward")
plt.plot(*ut_mean, "g^", markersize=9, label="unscented: nails it")
plt.legend(fontsize=8); plt.axis("equal"); plt.title("Banana problem: curvature defeats the tangent line")
plt.tight_layout(); plt.show()
print(f"mean estimates — MC {mc_mean.round(3)}, EKF {ekf_mean.round(3)}, UT {ut_mean.round(3)}")

mean estimates — MC [-0.005  0.94 ], EKF [0. 1.], UT [0.    0.941]


/tmp/ipykernel_2045392/3712700104.py:29: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.tight_layout(); plt.show()


**What just happened.** Three estimates of the same mean: Monte Carlo over 4000 samples gives $[-0.005, 0.940]$, the unscented transform gives $[0.000, 0.941]$ from **five** points, and the EKF-style linearization insists on $[0.000, 1.000]$.

And we can do better than comparing against Monte Carlo, because this problem has an exact answer. With $\theta \sim \mathcal{N}(\pi/2, \sigma^2)$ and $\sigma = 0.35$, the expected $y$-coordinate is $E[r]\,E[\sin\theta] = \sin(\pi/2)\,e^{-\sigma^2/2} = e^{-0.06125} = \mathbf{0.9406}$. So the unscented transform's 0.941 is right to three decimals, the Monte Carlo 0.940 is right to the precision 4000 samples can offer, and the EKF's 1.000 is wrong by **5.9%** — a bias, not noise, and one that no amount of extra data would reduce.

**Why linearization is biased rather than merely inaccurate.** The tangent line at the mean is a straight map, and a straight map sends the centre of a distribution to the centre of its image. But the true map is curved: the cloud bends into an arc, and the mean of an arc lies *inside* the curve, not on it. This is Jensen's inequality wearing a geometric hat — $E[\sin\theta] < \sin(E[\theta])$ for a symmetric spread around a concave stretch. No first-order method can see this, because the effect is second-order in the spread, which is exactly what "EKF is accurate to first order, UKF to second" means in practice.

**The efficiency is the striking part.** Five deterministic points matched a 4000-sample Monte Carlo estimate — and beat it, since MC's $-0.005$ in the $x$-coordinate is sampling noise around a true value of exactly 0. Sigma points are a quadrature rule, chosen to reproduce the mean and covariance exactly and to capture the leading curvature correction, not a small random sample. Run this cell twice and the UT answer is identical while the MC answer moves.

**What this does not fix.** The unscented transform still summarises the outcome as one mean and one covariance — it fits a single Gaussian to the mapped points. Look at the scatter: the truth is a banana, and *no* Gaussian describes a banana well, whatever its mean. UKF gets the location right and still misrepresents the shape. When the belief is not merely curved but genuinely multi-hypothesis, one blob is not a bad approximation but the wrong data structure, which is where Session 3 begins.

---
### 🕐 Session 3 of 3 — *Particle Filters* (~40 min)
**Goal:** represent ANY belief with weighted samples; track through multimodality.
**Builds on:** Session 2.

---

<details>
<summary>🎓 <b>Teacher notes — Session 3: Particle Filters</b></summary>

**Timing (~40 min).** 8 min why a Gaussian cannot represent the question · 12 min the predict/reweight/resample loop · 12 min the corridor demo and its three snapshots · 8 min costs, the curse of dimensionality, and the escalation rule.

**Open with a question the previous two sessions cannot answer.** A robot in a corridor measures only its distance to the centre. It could be two metres left or two metres right, and those are equally consistent with everything observed. Ask the room to draw the honest belief. It is two bumps — and then ask what a Kalman filter would report. A single Gaussian centred at zero, which is the one place the robot certainly is *not*. The failure is representational, not a matter of tuning, and that framing motivates the whole session.

**Board first — three verbs.** Predict (push every particle through the dynamics, with noise), reweight (score each by measurement likelihood), resample (duplicate the plausible, discard the rest). Write those three words and map each to its line in the code. The loop is short and students can hold all of it, which is rare and worth exploiting.

**Why resampling exists.** Without it, weights concentrate on a handful of particles within a few steps — most of the population ends up with a weight of ~1e-12, consuming compute while contributing nothing. That is *degeneracy*, and the effective sample size `1/Σw²` in the code is how we detect it. Point at the `< Np/2` threshold: we resample only when the effective population has halved, because resampling is not free either — it discards diversity and adds its own noise. Students often assume resampling every step is more correct; it is not, and knowing why is a genuine insight.

**Misconception.** "Particles are just Monte Carlo sigma points." They are categorically different. Sigma points are a deterministic quadrature rule for a distribution *assumed Gaussian*; particles are a sampled representation of an *arbitrary* distribution, with no assumed shape. That is the entire escalation — UKF handles curvature within one blob, particles handle any number of blobs of any shape.

**Ask the room.** "The robot drifts steadily rightward. Why does that eventually resolve the ambiguity?" Let them work it. Both clusters drift right, but the true one keeps matching the measurements while the mirror cluster predicts $|x|$ evolving the wrong way — it drifts toward the wall as the truth drifts away from it. The impostor's likelihoods fall, its weights collapse, and resampling starves it out. Motion breaks a symmetry that no single measurement could.

**Point at the snapshots in order.** At $t=2$ two clean modes: the filter honestly does not know. At $t=20$ still two, with one thinning. At $t=55$ one mode on the truth. Say explicitly that the middle panel is the filter being *right*, not confused — reporting genuine ambiguity is the correct behaviour, and a filter that collapsed to one mode early would be confidently wrong half the time.

**Do not oversell — give the costs their airtime.** $O(N_p)$ per step with $N_p = 3000$ here for one dimension. The curse is severe: particles needed grows roughly exponentially in state dimension, so plain particle filtering is impractical much beyond about 3–5 dimensions without extra structure (Rao–Blackwellisation, better proposals). End on the escalation rule: EKF if mildly nonlinear, UKF if curvy or Jacobian-hostile, particles only if multimodal or seriously non-Gaussian. Never more machinery than the problem demands.
</details>

## 4. When the Belief Isn't a Blob

💡 **Intuition.** EKF/UKF still summarize belief as mean + covariance — one Gaussian blob. Some problems are **multimodal**: a robot that observes only its *distance* to a wall genuinely can't distinguish left from right, and the honest belief is two blobs. The particle filter drops the Gaussian religion: carry $N$ weighted samples (*particles*), move each through the dynamics (with noise), reweight by measurement likelihood, and **resample** to cull the walking dead. It's the [LLN](../Intro_Math/Analysis/Independence.ipynb) as a filter — any shape of belief, at Monte Carlo prices.

In [4]:
# 1-D corridor localization: robot observes |position| (symmetric!) plus noise
T2, Np = 60, 3000
x_true_pos = -2.0                                        # starts LEFT of center
traj, meas = [], []
pos = x_true_pos
for t in range(T2):
    pos += 0.05 + rng.normal(0, 0.02)                    # drifts rightward
    traj.append(pos); meas.append(abs(pos) + rng.normal(0, 0.1))

particles = rng.uniform(-4, 4, Np)                       # know nothing
weights = np.ones(Np)/Np
snapshots = {}
for t in range(T2):
    particles += 0.05 + rng.normal(0, 0.05, Np)          # predict
    lik = np.exp(-(meas[t] - np.abs(particles))**2 / (2*0.1**2))
    weights = lik + 1e-300; weights /= weights.sum()     # update
    if 1/np.sum(weights**2) < Np/2:                      # resample when degenerate
        particles = particles[rng.choice(Np, Np, p=weights)]
        weights = np.ones(Np)/Np
    if t in (2, 20, 55): snapshots[t] = particles.copy()

fig, axes = plt.subplots(1, 3, figsize=(10, 2.6), sharey=True)
for ax, (t, p) in zip(axes, snapshots.items()):
    ax.hist(p, bins=80, range=(-4, 4), density=True)
    ax.axvline(traj[t], color="r", linestyle=":", label="truth")
    ax.set_title(f"t={t}: {'two hypotheses!' if t < 30 else 'disambiguated'}")
    ax.legend(fontsize=7)
plt.suptitle("|x| measurements: belief is honestly bimodal until motion breaks the symmetry", y=1.04)
plt.tight_layout(); plt.show()

/tmp/ipykernel_2045392/4274007459.py:29: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.tight_layout(); plt.show()


**What just happened.** Three snapshots of a belief that changes *shape*, not just position. At $t=2$ the particles form two clean clusters near $\pm 2$; at $t=20$ two are still there with one visibly thinning; by $t=55$ a single cluster sits on the truth. The filter began knowing nothing — particles spread uniformly over the whole corridor — and was never told the robot started on the left.

**The middle panel is the filter being right.** This is the part worth insisting on. Two modes is not confusion or a bug: the measurement is $|x|$, so positions $+2$ and $-2$ are *exactly* equally consistent with everything observed so far, and any filter reporting a single confident answer at $t=2$ would be wrong half the time it was run. The particle filter's advantage here is not accuracy, it is honesty about what the data does and does not determine.

**How the ambiguity breaks.** Both clusters drift rightward, because the motion model applies to every particle alike. But the true robot moves *away* from the centre while its mirror image moves *toward* it, so the two hypotheses predict $|x|$ evolving in opposite directions. Within a few steps the impostor's predicted measurements diverge from the real ones, its likelihoods collapse, and `rng.choice` stops selecting it. Motion resolved a symmetry that no single measurement could — the same lesson as the pendulum in Session 1, now with the ambiguity discrete rather than continuous.

**Watch the two lines doing the real work.** `weights = lik + 1e-300` prevents an all-zero weight vector from producing a division by zero when every particle is implausible — a genuine failure mode called particle depletion, not defensive padding. And `1/np.sum(weights**2) < Np/2` is the effective sample size test: resample only once the effective population has halved, because resampling costs diversity and injects its own noise. Resampling every step is a common beginner instinct and it measurably degrades the estimate.

**The bill.** Three thousand particles to localise in *one* dimension. Particle counts scale roughly exponentially with state dimension, so plain particle filtering is generally impractical past three to five dimensions without additional structure — Rao–Blackwellisation to handle the linear substructure analytically, or smarter proposal distributions. That is why the escalation rule ends where it does: reach for particles when the belief genuinely is not a blob, and not before. Here it was not a blob, and nothing simpler could have represented the question.

The disambiguation: both hypothesis clusters drift right, but only one keeps matching the measurements as the true robot crosses regions where $|x|$ evolves differently — the impostor cluster starves and dies at resampling. **No Gaussian filter can even represent the question.**

Costs to respect: $O(N_p)$ per step, particle death in high dimensions (the curse), and resampling noise. The escalation rule: EKF if mildly nonlinear, UKF if curvy or Jacobian-hostile, particles if multimodal or seriously non-Gaussian — never more machinery than the problem demands.

## 5. Conclusion

Linearize, sigma-sample, or particle-sample: three ways to keep the predict/update heartbeat when the world stops being linear. You now own the full state-estimation ladder from LMS to Monte Carlo.

---
## Where next

- [Recurrent Neural Networks](./Intro_RNN.ipynb) — *learn* the dynamics instead of modeling them.
- [Uncertainty in ML](../Intro_Mach_Learn/Uncertainty_in_ML.ipynb) — ensembles: particle filtering's spirit in deep learning.